# Producción Agricola. (Preprocesamiento)

En la parte anterior descargamos y la hicimos minimimamente usable los datos del DGSIAP. Ahora dejaremos de lado la gestion de los datos y nos centraremos en los datos mismos.

In [201]:
# Obtenemos los datos a procesar
from scripts.config import PRELOADS
import pickle

with open(PRELOADS['AGROPROD_0'], "rb") as preprocess: 
    dgsiap = pickle.load(preprocess)

# Verificamos que los archiovs se encuentren en el archivo precargado
dgsiap.root

## Carga desde raw.

Ahora vamos a cargar los archivos desde la carpeta /raw

In [202]:
datadic = dgsiap.root

codebook = datadic.open()
codebook.head()

,Unnamed: 0.1,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Viendo que el diccionario de datos no esta bien formateado. Vamos a experimentar con él.

### Diccionario de Datos

In [203]:
import pandas as pd
ubi = datadic.ubication
# Quitamos rengolnes y elegimos el nombre de las variables como 
exc = pd.read_excel(ubi, skiprows=9, index_col=2)


# Introducimos el siguiente conjunto de paramentos
exc_pars = {"skiprows":9,
            "index_col":2}

df = pd.read_excel(ubi, **exc_pars)
# print(df.columns)
v = list(df.columns)[:2]
df.drop(columns=list(df.columns)[:2])

# Todo esto será guardado en un formato personalizado.

def preconfig(df):
    # Esto puede cambiar en función de la base de datos.
    subdf = df.iloc[-7:-1,1:2]
    delete_columns = list(df.columns)[:2]
    new_df = df.drop(columns=delete_columns)
    new_df = new_df.iloc[:-7]
    return subdf, new_df

# Usando la clase File

codebook = datadic.open(pars=[1, exc_pars])
vars, codes = preconfig(codebook)
print(vars)
codes

                    Consecutivo\nen la base
Nombre\nde campo                           
NaN                                     NaN
NaN                                T: Texto
NaN                       N:Numérico entero
NaN                     D: Numérico decimal
NaN                   #: Carácter numérico.
NaN               A: Carácter alfanumérico.


,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4.0,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Idestado,N,2.0,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Nomestado,T,60.0,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Idddr,N,3.0,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Nomddr,T,60.0,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Idcader,N,2.0,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Nomcader,T,90.0,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Idmunicipio,T,3.0,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Nommunicipio,T,45.0,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...


Del diccionario de datos obtuvimos dos elementos

* El catalogo: Donde vemos la información de las variables
* Las variables: Los tipos de datos utilizados en los datos.

Vamos a formatear los datos para usarlos más adelante.

In [204]:
vars = vars['Consecutivo\nen la base'].to_list()[1:]
guide_vars = {v.split(":")[0]: v.split(":")[-1].strip() for v in vars}
guide_vars

{'T': 'Texto',
 'N': 'Numérico entero',
 'D': 'Numérico decimal',
 '#': 'Carácter numérico.',
 'A': 'Carácter alfanumérico.'}

Además, simplificaremos el diccionario de datos mediante 2 elementos.
* Validación de Datos. Las primeras 3 columnas definen los lineamientos que deberian seguir los datos.
* Guia de uso. Una herramienta para la gente que no es experta en el tema. 

In [205]:
# La usaremos siempre que queramos consultar la información de una variable.

def prodagro_help(var):
    guide_book= codes[["Descripción", "Observaciones"]]
    return guide_book.loc[var]

### Correspondencia de Datos

Para nuestro analisis podemos definir nuestras propias reglas. Pero primero corroboraremos si los datos siguen sus propios lineamientos.

Todo esto lo haremos con uno del conjunto de datos.

In [206]:
test = dgsiap.branch[0]

csv_pars = {
    "low_memory":False,
    "index_col":0
}
test_df = test.open(pars=[1, csv_pars])
test_df

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.00,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.00,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.00,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.20,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.00,16.33,5285.21,2589752.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35897,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,7960000,Nuez,10.0,10.0,0.0,33.00,3.30,55000.00,1815000.00
35898,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,9130000,Uva,1.0,1.0,0.0,9.90,9.90,11444.38,113299.36
35899,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,15050000,Pastos y praderas,36.0,36.0,0.0,1396.96,38.80,963.14,1345470.00
35900,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,5050000,Agave,260.0,170.0,0.0,14960.00,88.00,1200.00,17952000.00


#### Columnas 

Primero verificaremos que las columnas coinciden.

In [207]:
test_columns = test_df.columns.tolist()
data_vars = codes.index.tolist()

# Vemos las columnas que son diferentes.
## Si los nombres son muy distintos puede ser que esten, pero en distinto orden, 
for test, real in zip(test_columns, data_vars):
    if test!= real:
        print(test, real)

Preciomediorural Precio


En este caso las columnas se llaman parecido. Pero el conjunto de datos le cambio el nombre. El contexto nos puede decir que son lo mismo. Pero el diccionario debe de corroborarlo. Por lo tanto, veremos la descripción de la variable

In [208]:
# Vemos la información asociada a la variable problemática. 
prodagro_help('Precio')

Descripción                                     Precio medio rural
Observaciones    Precio medio rural, la unidad de medida son pe...
Name: Precio, dtype: str

Vemos que el nombre del conjunto de datos es la descripción de la variable del diccionario. Entonces podemos hacer el cambio sin perder información. 

In [209]:
norm_df = test_df.copy()
norm_df.columns = codes.index
# Volvemos a hacer la verificación de columnas 
def detect_diff_columns(df, ref):
    df_cols = df.columns.tolist()
    ref_cols = ref.index.tolist()
    prob_cols = [(r, t) for r, t in zip(ref_cols, df_cols) if t!=r]
    return prob_cols

print("Base Original", detect_diff_columns(test_df, codes))
print("Base Normalizada", detect_diff_columns(norm_df, codes))

Base Original [('Precio', 'Preciomediorural')]
Base Normalizada []


Esto hay que hacerlo con cada conjunto de datos. Esto para asegurar que las columnas reflejan lo definido en el diccionario de datos.

In [210]:
def tree_cols(tree, tpars, tpreconfig):
    ref = tree.root
    files = tree.branch
    # Process Root File.
    codebook = ref.open(pars=[1, tpars['root']])
    _ , catalog = tpreconfig['root'](codebook)
    for f in files:
        df = f.open(pars=[1, tpars['branch']])
        diff = detect_diff_columns(df, catalog)
        if len(diff)>0:
            print(diff)

dgsiap_pars = {
    "root": exc_pars,
    "branch": csv_pars
}

dgsiap_preconfig = {
    'root': preconfig,
    'branch': None
}

tree_cols(dgsiap, dgsiap_pars, dgsiap_preconfig)

[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]


Vemos que en todo el arbol hay dos columnas problematicas. Entonces primero iremos a algun conjunto de datos con esa columna cambiada.  

In [211]:
# Buscamos alguna con el cambio. 
diff_test = dgsiap.branch[7]
print(diff_test.filename)
csv_pars = {
    "low_memory":False,
    "index_col":0
}
diff_df = diff_test.open(pars=[1, csv_pars])
detect_diff_columns(diff_df, codes)


Cierre_agricola_mun_2018.csv


[('Nomcultivo', 'Nomcultivo Sin Um')]

De forma similar haremos el analisis de la columna problematica.

In [212]:
print(prodagro_help('Nomcultivo')['Descripción'])
prodagro_help('Nomcultivo')['Observaciones']

Nombre del cultivo agrícola


'Nombre asignado en el catálogo de cultivos de la DGSIAP.'

Esta vez la descripción no nos dio información. Intentaremos ver si el segundo nombre se encuentra.

In [213]:
# Esta linea puede dar un error. Pero es lo que nos interesa en este momento.
try:
    prodagro_help('Nomcultivo Sin Um')
except KeyError as e:
    print("Esta columna no esta en el diccionario")
print(codes.index)


Esta columna no esta en el diccionario
Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo',
       'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida',
       'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada',
       'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio',
       'Valorproduccion'],
      dtype='str', name='Nombre\nde campo')


Estas inconsistencias requieren un analisis más profundo en todo el conjunto de datos. Esto porque la explicación viene provenida en otra variable. Rendimiento.

In [214]:
print(prodagro_help('Rendimiento')['Descripción'])
prodagro_help('Rendimiento')['Observaciones']

Rendimiento


'Rendimiento, la unidad de medida son toneladas por hectárea, con excepción de los cultivos que tienen otra métrica la cual se señala en el nombre del cultivo (gruesa, manojo, planta, entre otros).'

Entonces vemos que el diccionario indicaba que el nombre cultivo incluia la unidad especial que se utilizaba. Aqui haremos la verificación pero no modificaciones. Esta clase de modificaciones implican cambiar las observaciones y eso queda fuera de la correspondencia. 

Para ello veremos si las observaciones cambiaron de nombre al cambiar la columna.

In [215]:
# Seleccionamos un conjunto con y sin el problema.
## Sin el cambio 
print(test_df.columns)
## Con el cambio 
print(diff_df.columns)

Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo',
       'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida',
       'Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada', 'Cosechada',
       'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Preciomediorural',
       'Valorproduccion'],
      dtype='str')
Index(['Anio', 'Idestado', 'Nomestado', 'Idddr', 'Nomddr', 'Idcader',
       'Nomcader', 'Idmunicipio', 'Nommunicipio', 'Idciclo',
       'Nomcicloproductivo', 'Idmodalidad', 'Nommodalidad', 'Idunidadmedida',
       'Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada', 'Cosechada',
       'Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio',
       'Valorproduccion'],
      dtype='str')


In [216]:
# Ahora veremos los valores unicos de la columna Noumunidad de ambas bases de datos
print(test_df['Nomunidad'].unique().tolist())
print(diff_df['Nomunidad'].unique().tolist())

# Verificamos nuestra hipotesis.
diff_plants = diff_df[diff_df['Nomunidad'] == "Planta"]
print(diff_plants['Nomcultivo Sin Um'].unique())

test_plants = test_df[test_df['Nomunidad'] == "Planta"]
print(test_plants['Nomcultivo'].unique())

['Tonelada', 'Metro cuadrado', 'Gruesa', 'Planta', 'Manojo', 'Miles de lts.']
['Tonelada', 'Metro cuadrado', 'Gruesa', 'Planta', 'Manojo', 'Miles de lts.']
<StringArray>
[   'Palma de ornato', 'Viveros de durazno', 'Viveros de manzana',
    'Viveros de nuez',           'Calancoe',          'Cineraria',
            'Geranio',         'Nochebuena',   'Árbol de navidad',
           'Cyclamen',   'Lilium (azucena)',        'Pensamiento',
            'Petunia',   'Tulipán holandés',              'Dalia',
           'Aretillo',            'Begonia',              'Belén',
         'Crisantemo',               'Rosa',     'Zempoalxochitl',
           'Orquídea',              'Fresa',          'Hortensia',
 'Plantero de tabaco',  'Plantas de ornato']
Length: 26, dtype: str
<StringArray>
[   'Palma de ornato (planta)',  'Plantas de ornato (planta)',
          'Viveros de durazno',          'Viveros de manzana',
             'Viveros de nuez',           'Calancoe (planta)',
            'Geranio (p

Concluyendo que la hipotesis es cierta. El cambio de nombre fue por un cambio en la escritura de las observaciones. Lo ideal sería hacer la normalización ahora. Sin emabargo, modificar los datos queda fuera de la etapa de correspondencia. Este cambio se hará cuando entremos a la limpieza de los datos (\raw a \interm). Finalmente hacemos la normalización permitida 

In [217]:
def precio_rule(df, ref):
    df.columns = ref.index
    return df

norm_corr = {
    "Precio": lambda x,y : precio_rule(x, y)
}

def tree_normcols(tree, tpars, tpreconfig, **kwargs):
    norm_rules = kwargs.get("norm_rules", None)
    ref = tree.root
    files = tree.branch
    # Process Root File.
    codebook = ref.open(pars=[1, tpars['root']])
    _ , catalog = tpreconfig['root'](codebook)
    new_branch = []
    for f in files:
        # Process Branch Files
        df = f.open(pars=[1, tpars['branch']])
        diff = detect_diff_columns(df, catalog)
        if len(diff)>0:
            print(diff)
            if norm_rules is not None:
                for d in diff:
                        if d == ("Precio", "Preciomediorural"):
                            new_branch.append(precio_rule(df, catalog))
                        else:
                            new_branch.append(df)
    return new_branch

newb = tree_normcols(dgsiap, dgsiap_pars, dgsiap_preconfig, norm_rules=norm_corr)

[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Precio', 'Preciomediorural')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]
[('Nomcultivo', 'Nomcultivo Sin Um')]


In [218]:
# Hacemos una verificación de las columnas 
for n in newb:
    print(n.columns.tolist()[-5:])

['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']
['Siniestrada', 'Volumenproduccion', 'Rendimiento', 'Precio', 'Valorproduccion']


Una vez normalizamos las columnas podemos guardarlo como un objeto. 

In [232]:
from scripts.Files import File, FileTree
from scripts.config import DATA_DIR

def prodagro_help(var):
    guide_book= codes[["Descripción", "Observaciones"]]
    return guide_book.loc[var]


def normtree(tree: FileTree, file_pars: dict, file_preconfig: dict, norm_rules: dict):
    interm_path = DATA_DIR / "interm" / tree.source
    root_file = tree.root
    branch_files = tree.branch
    codebook = root_file.open(pars=[1, file_pars['root']])
    vars_f , codes = file_preconfig['root'](codebook)
    gvars = vars_f['Consecutivo\nen la base'].tolist()
    guide_vars = {v.split(":")[0]: v.split(":")[-1].strip() for v in gvars[1:]}
    root_ext = root_file.filename.split(".")[-1]
    if root_ext == "csv":
        codes.to_csv(interm_path / root_file.filename)
    else:
        codes.to_excel(interm_path / root_file.filename)
    bfiles = []
    for file in branch_files:
        df = file.open(pars=[1, file_pars['branch']])
        diff = detect_diff_columns(df, codes)
        if len(diff) > 0:
            if norm_rules is not None:
                for d in diff:
                    try:
                        ndf = norm_rules[d](df, codes)
                    except:
                        ndf = df.copy()
        else:
            ndf = df
        filename = file.filename
        ext = filename.split(".")[-1]
        if ext == "csv":
            ndf.to_csv(interm_path / filename)
        else:
            ndf.to_excel(interm_path / filename)
        normfile = File(interm_path / filename, tree.source)
        bfiles.append(normfile)
    normcodes = File(interm_path / root_file.filename, tree.source)
    ntree = FileTree(normcodes, bfiles, tree.source)
    print(guide_vars)
    ntree.add_subroot(guide_vars) 
    return ntree


dgsiap_apars = {
    "root": exc_pars,
    "branch": csv_pars
}

dgsiap_apreconfig = {
    'root': preconfig,
    'branch': None
}

dgsiap_normrules = {
    ("Precio", "Preciomediorural"): lambda x, y: precio_rule(x, y)   
}

norm_dgsiap = normtree(dgsiap, dgsiap_apars, dgsiap_apreconfig, dgsiap_normrules)







{'T': 'Texto', 'N': 'Numérico entero', 'D': 'Numérico decimal', '#': 'Carácter numérico.', 'A': 'Carácter alfanumérico.'}
[<scripts.Files.File object at 0x7f75263883b0>, <scripts.Files.File object at 0x7f7526388530>, <scripts.Files.File object at 0x7f751ede1550>, <scripts.Files.File object at 0x7f7526388290>, <scripts.Files.File object at 0x7f751ede1cd0>, <scripts.Files.File object at 0x7f75263884d0>, <scripts.Files.File object at 0x7f7525c3d970>, <scripts.Files.File object at 0x7f7525e4d6d0>, <scripts.Files.File object at 0x7f752784ab10>, <scripts.Files.File object at 0x7f7507bde210>, <scripts.Files.File object at 0x7f751ede16d0>, <scripts.Files.File object at 0x7f752784b6b0>, <scripts.Files.File object at 0x7f7507bde030>, <scripts.Files.File object at 0x7f751ede2690>, <scripts.Files.File object at 0x7f7525c3e990>, <scripts.Files.File object at 0x7f7526389610>, <scripts.Files.File object at 0x7f751ede1d90>, <scripts.Files.File object at 0x7f7526388d70>, <scripts.Files.File object at 0

A partir de este momento trabajaremos con los datos del DGSIAP normalizados. Nomás no olvidemos el paso pendiente
* Revisar la columna NomCultivo en la limpieza de datos.

In [220]:
# Guardamos los datos normalizados 
from scripts.config import ROOT
with open(ROOT / "preload" /"DGSIAP_NORM.pkl", "wb") as datanorm:
    pickle.dump(norm_dgsiap, datanorm)

#### Reglas.

El diccionario de datos definio algunas reglas previamente. Vamos a revisar que se cumplan.

In [221]:
codebook = norm_dgsiap.root
# Las columnas Tipo, Long y Rangos\nValidos definen las reglas establecidas por el DGSIAP.
codes = codebook.open(pars=[1, {"index_col":0}])
codes

,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Idestado,N,2,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Nomestado,T,60,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Idddr,N,3,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Nomddr,T,60,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Idcader,N,2,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Nomcader,T,90,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Idmunicipio,T,3,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Nommunicipio,T,45,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...


Para ello crearemos funciones que nos permitan validar si una columna cumple la regla establecida. Para esto lo trabajaremos mediante el tipo de dato.

##### Variables Categoricas.

Las variables categoricas son aquellas que poseenn un conjunto finito de valores. El diccionario lo denota mediante la forma "{a,...,b}"

In [222]:
import re

# Crearemos un regex para detectarlo.
var = "{1,...,32}"
print(var)
g = re.search(r"{(.*)}", var)
print(g.group(1))  # Con esto lo obtendremos.
cat_pattern = r"{(?:\d.*)}"
cat_vars = codes.loc[codes['Rangos\nválidos'].str.contains(cat_pattern, regex=True, na=False)]
cat_vars


{1,...,32}
1,...,32


,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Idestado,N,2,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Idddr,N,3,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Idcader,N,2,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Idmunicipio,T,3,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Idciclo,N,1,"{1,….,3}",Código que identifica al ciclo agrícola,Código asignado en el catálogo de ciclo agrícola.
Idmodalidad,N,1,"{1,2}",Código que identifica al tipo de modalidad híd...,Clave asignada en el catálogo de modalidad híd...
Idunidadmedida,N,1,"{1,….,6}",Código que identificala unidad de medida,Código asignado en el catálogo de unidad de me...
Idcultivo,N,3,"{1,….,410}",Código que identifica al cultivo agrícola,Clave asignado en el catálogo de cultivos de l...


Vemos que el DGSIAP considera los valores categóricos como numeros enteros. A excepción de IdMunicipio. Que por el contexto del diccionario parece más un error de escritura que algo con fundamento. Lo anotaremos para el resumen de correspondencia de datos. Pero eventualmente haremos el cambio a numeros enteros o a categoricos directamente.

In [223]:
def categorical_guide(value, avaib_range, **kwargs):
    cat_pattern = kwargs.get("catp", r"{(.*)}")
    av_range = re.search(cat_pattern, avaib_range)
    r_g = av_range.group(1).split(",")
    m, mm = int(r_g[0].strip()), int(r_g[-1].strip())
    # print(m, mm, value, type(value))
    values_space = set(range(m, mm + 1))
    return value in values_space


categotical_rules = {catname: lambda x, y=cat_vars.loc[catname, 'Rangos\nválidos']: categorical_guide(x, y) for catname in cat_vars.index}

# Antes de seguir quitaremos las variables previamente utilizadas para evitar sobreescritura de reglas.
rest_codes =  codes.loc[~codes['Rangos\nválidos'].str.contains(cat_pattern, regex=True, na=False)]
rest_codes

,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Nomestado,T,60,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Nomddr,T,60,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Nomcader,T,90,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Nommunicipio,T,45,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...
Nomcicloproductivo,T,40,{CCC},Nombre del ciclo agrícola,"Nombre que identifica al ciclo agrícola, gener..."
Nommodalidad,T,20,{CCC},Nombre de la modalidad hídrica,Nombre de la modalidad hídrica que hace refere...
Nomunidad,T,40,{CCC},Nombre que identificala unidad de medida,Nombre asignado en el catálogo de unidad de me...
Nomcultivo,T,40,{CCC},Nombre del cultivo agrícola,Nombre asignado en el catálogo de cultivos de ...


##### Variables tipo Texto (T) y Numero Decimal. 

Como esto es algo más directo haremos las dos al mismo tiempo. Ya que en ambos casos, pero de forma distinta, hay que revisar si estan dentro del tamaño establecido.

Notemos que el diccionario de datos es inconsistente en los valores tipo decimal. Ya que hay variables, como Sembrada, que la longitud considera los decimales y variables, como Precio, que no la consideran. En este caso, como las variables son mediciones no tienen una reestricción forzada en sus valores, más alla de ser valores positivos. Entonces aplicaremos la regla contando los decimales.

In [224]:
# Variable Texto usa la longitud.

text_vars = rest_codes[rest_codes["Tipo"] == "T"]
decimal_vars = rest_codes[rest_codes["Tipo"] == "D"]
num_vars = rest_codes[rest_codes["Tipo"] == "N"]


## Primero el caso del año porque es tipo entero pero esta reglamentado como si fuera texto.
### Esto es para indicar que el año debe escribirse en su forma completa y no resumida. Por ejemplo, 2005 en lugar de 05.


def text_guide(value, long):
    return True if len(value) <= long else False


def decimal_guide(value, avaib_range, left = True, right = True, plus=True):
    digits = avaib_range[1:-1]
    number = digits.split(".")
    ints = len(number[0])
    decs = len(number[1]) if len(number) == 2 else 0
    mm = 10 ** (ints) - 1
    if plus:
        m = 0
    else: 
        m = - mm
    check = -1
    if left:
        check = m <= value <= mm if right else  m <= value < mm
    else:
        check = m < value <= mm if right else  m < value < mm
    check = True if check and value - round(value, decs) == 0 else False
    return check


def dec_str_guide(value, long):
    num_code = "#"*long
    num_code = "{" + num_code + "}"
    return decimal_guide(value, num_code)

text_rules = {catname: lambda x, y=text_vars.loc[catname, 'Long.']:text_guide(x, y) for catname in text_vars.index}
decimal_rules = {catname: lambda x, y=decimal_vars.loc[catname, 'Rangos\nválidos']: decimal_guide(x, y) for catname in decimal_vars.index}
decstr_rules = {catname: lambda x, y=num_vars.loc[catname, 'Long.']: dec_str_guide(x,y) for catname in num_vars.index}



Ahora probaremos las reglas en un conjunto de datos de prueba.

In [225]:
datatest = norm_dgsiap.branch[0]
test = datatest.open(pars=[1, {"index_col":0}])
test

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.00,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.00,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.00,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.20,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.00,16.33,5285.21,2589752.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35897,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,7960000,Nuez,10.0,10.0,0.0,33.00,3.30,55000.00,1815000.00
35898,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,9130000,Uva,1.0,1.0,0.0,9.90,9.90,11444.38,113299.36
35899,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,15050000,Pastos y praderas,36.0,36.0,0.0,1396.96,38.80,963.14,1345470.00
35900,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,5050000,Agave,260.0,170.0,0.0,14960.00,88.00,1200.00,17952000.00


Despues de veriifcar que todos las columnas definidas en el diccionario poseen una regla asociada hacemos la verificación.

In [226]:
rules = {**categotical_rules, **text_rules,**decimal_rules, **decstr_rules}

var_not_rules = [v for v in test.columns if not v in set(rules.keys())]
print("Variables sin regla asignada", var_not_rules)


verf_rules = test.copy()
for c in test.columns:
    verf_rules[c]=test[c].apply(rules[c])

verf_rules



Variables sin regla asignada []


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
0,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35897,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
35898,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
35899,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True
35900,True,True,True,True,True,True,True,True,True,True,...,True,False,True,True,True,True,True,True,True,True


Para verlo de forma más clara. Contaremos el numero de valores verdadero por columna. 

In [227]:
count_true = verf_rules.sum(axis=0)
columns_prob = count_true[count_true < verf_rules.shape[0]]
columns_prob

Nommunicipio      35839
Idunidadmedida        0
Idcultivo             0
Rendimiento       35487
Precio            35487
dtype: int64

Vemos que hay 6 columnas que no siguen los lineameintos establecidos por el diccionario. Entonces debemos obtener más información.

In [228]:
data_problems = {}
for prob in columns_prob.index:
    prob_sector = verf_rules[verf_rules[prob]==False]
    prob_values = test.loc[prob_sector.index, columns_prob.index]
    data_problems[prob] = prob_values

data_problems.keys()

dict_keys(['Nommunicipio', 'Idunidadmedida', 'Idcultivo', 'Rendimiento', 'Precio'])

Ya identificamos las columnas problematicas y podemos ver el motivo de los fallos. 

In [229]:
data_problems['Nommunicipio'].head(n=3)  # Nombres de municipios muy largos.

,Nommunicipio,Idunidadmedida,Idcultivo,Rendimiento,Precio
4954,Dolores Hidalgo Cuna de la Independencia Nacional,200201,5070000,10.25,28575.85
4955,Dolores Hidalgo Cuna de la Independencia Nacional,200201,5370000,25.15,10148.42
4956,Dolores Hidalgo Cuna de la Independencia Nacional,200201,5490000,21.85,663.15


In [230]:
print(codes.loc['Idunidadmedida'])
print(data_problems['Idunidadmedida'].head(n=3)) # Si vemos el diccionario de datos notamos que dice tamaño 1 y estos poseen 6 digitos.
print(codes.loc['Idcultivo'])
data_problems['Idcultivo'].head(n=3)  # En este caso ocurre algo similar.

Tipo                                                               N
Long.                                                              1
Rangos\nválidos                                             {1,….,6}
Descripción                 Código que identificala unidad de medida
Observaciones      Código asignado en el catálogo de unidad de me...
Name: Idunidadmedida, dtype: object
     Nommunicipio  Idunidadmedida  Idcultivo  Rendimiento   Precio
0  Aguascalientes          200201    5490000        23.50   654.00
1  Aguascalientes          200201    5670000        17.00  6887.78
2  Aguascalientes          200201    5490000        22.42   645.00
Tipo                                                               N
Long.                                                              3
Rangos\nválidos                                           {1,….,410}
Descripción                Código que identifica al cultivo agrícola
Observaciones      Clave asignado en el catálogo de cultivos de l...
Name: 

,Nommunicipio,Idunidadmedida,Idcultivo,Rendimiento,Precio
0,Aguascalientes,200201,5490000,23.50,654.00
1,Aguascalientes,200201,5670000,17.00,6887.78
2,Aguascalientes,200201,5490000,22.42,645.00


In [231]:
print(codes.loc['Rendimiento'])
print(data_problems['Rendimiento'].isna().sum()) # Ambos casos se limitan a valores nulos.
print(codes.loc['Precio'])
data_problems['Precio'].isna().sum()

Tipo                                                               D
Long.                                                             10
Rangos\nválidos                                       {########.###}
Descripción                                              Rendimiento
Observaciones      Rendimiento, la unidad de medida son toneladas...
Name: Rendimiento, dtype: object
Nommunicipio        0
Idunidadmedida      0
Idcultivo           0
Rendimiento       415
Precio            415
dtype: int64
Tipo                                                               D
Long.                                                              7
Rangos\nválidos                                        {#######.###}
Descripción                                       Precio medio rural
Observaciones      Precio medio rural, la unidad de medida son pe...
Name: Precio, dtype: object


Nommunicipio        0
Idunidadmedida      0
Idcultivo           0
Rendimiento       415
Precio            415
dtype: int64

# Conclusiones.

Revisando la coherencia y las reglas definidas por el diccionario. Mencionamos los hechos destacados.

* Coherencia. Existen columnas que poseen nombres diferentes a lo largo del conjunto de datos.
    * La columna Preciomediorural fue cambiada por la establecida en el diccionario.
    * La columna Cultivo sin Um se mantiuvo pero cambiará durante en analisis de datos.
* Seguimiento de Reglas. Varias columnas no siguen los lineamientos del diccionario por motivos menores.
    * Nommunicipio: Posee municipios con nombres muy largos.
    * Idunidadmedida, Idcultivo. Falta de claridad de parte del diccionario de datos.
    * Rendimiento, Precio: Columna con valores nulos.

Los datos tienen problemas según sus propias definiciones y lineamientos. Sin embargo, son elementos menores que serán trabajados en el procesamiento de datos.

Con este bloque terminamos el preprocesamiento, un analisis de los datos sin modificar los datos. A partir de la siguiente etapa vamos a comenzar con el procesamiento de datos, la preparación de datos para el analisis exploratorio de datos.